In [ ]:
import re
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from colorama import Fore,Style

from pathlib import Path
import os
import pandas as pd
import json
from System_prompts import*
from Data_soting_utils import*
import ast

converter = DocumentConverter()
chunker = HybridChunker()



In [ ]:
from LLM_Call import*

In [ ]:
folder_path="/localdata/user/kata_du/Automated Literature Survey/downloads/Test_Folder"

folder = Path(folder_path)

output_dir = folder / "References_JSON_Files"

file_path="/localdata/user/kata_du/Automated Literature Survey/downloads/Test_Folder/pdf_files/2020-A systematic literature review of cross-domain mod.pdf"

excel_log_path = os.path.join(output_dir, "Ref_process_log.xlsx")
output_dir.mkdir(exist_ok=True)
pdf_name = Path(file_path).name
output_path = os.path.join(output_dir, f"test_id_References.json")


try:
    # 1. Extraction
    doc = converter.convert(file_path).document
    chunks = chunker.chunk(dl_doc=doc)


except Exception as e:
    print(Fore.RED + f"Process Error: {e}" + Style.RESET_ALL)
    import traceback
    traceback.print_exc()


In [ ]:
def repair_truncated_json(raw_text):
    """
    Robust repair for potentially truncated JSON arrays.
    """
    # Normalize whitespace and remove BOM
    text = re.sub(r'\uFEFF', '', raw_text.strip())

    # Try parsing as-is
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"Parse failed: {e}")

    # Find potential start of array
    start_idx = text.find('[')
    if start_idx == -1:
        raise ValueError("No JSON array found")
    text = text[start_idx:]

    # Balance braces from end: find matching ] for [
    brace_count = 0
    last_valid_end = -1
    for i, char in enumerate(reversed(text)):
        if char == ']':
            brace_count += 1
        elif char == '[':
            brace_count -= 1
            if brace_count == 0:
                last_valid_end = len(text) - i
                break

    if last_valid_end == -1:
        raise ValueError("Could not balance JSON array")

    # Take up to balanced end, remove trailing comma
    repaired = text[:last_valid_end]
    repaired = re.sub(r',\s*([}\]])', r'\1', repaired)

    try:
        result = json.loads(repaired)
        print(f"Repaired to {len(result)} objects")
        return result
    except json.JSONDecodeError as e:
        raise ValueError(f"Repair failed: {e}")


def save_references_to_json(ai_response_text, file_path):
    """
    Parses the AI response and appends/saves it to a specific file path.
    
    Args:
        ai_response_text (str): The raw string output from the AI.
        file_path (str): The full path to the .json file (e.g., 'data/results.json').
    """
    try:
        # 1. Clean Markdown formatting if present
        cleaned_response = ai_response_text.strip()
        if cleaned_response.startswith("```"):
            cleaned_response = cleaned_response.splitlines()[1:-1]
            cleaned_response = "\n".join(cleaned_response)
            safe_json_str = repair_truncated_json(cleaned_response)
            
        new_entries = json.loads(safe_json_str)
        
        # Ensure new_entries is a list for consistency
        if not isinstance(new_entries, list):
            new_entries = [new_entries]

        # 2. Ensure the directory exists
        directory = os.path.dirname(file_path)
        if directory and not os.path.exists(directory):
            os.makedirs(directory)

        # 3. Handle file appending or creation
        final_data = []
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    existing_content = json.load(f)
                    final_data = existing_content if isinstance(existing_content, list) else [existing_content]
                except json.JSONDecodeError:
                    # If file is empty or corrupted, start fresh
                    final_data = []

        # Combine old data with new data
        final_data.extend(new_entries)

        # 4. Save to the specified path
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(final_data, f, indent=4, ensure_ascii=False)
        
        print(f"Successfully updated: {file_path}")

    except json.JSONDecodeError as e:
        print(f"Failed to parse AI response as JSON: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")



def process_references_from_chunks(chunks, doc,output_path):

    Ref_Keywords=["references", "bibliography", "reference list", 
                   "literature cited", "works cited", "sources", 
                   "literatur", "références"]

    for i, chunk in enumerate(chunks):
        chunk_heading = extract_chunk_heading(chunk)
        if any(key in chunk_heading.lower()for key in Ref_Keywords):
            Raw_text = chunk.text

            prompt_ref_text = f"Raw text:\n {Raw_text}"
            print(Fore.YELLOW +  f"\n --- {prompt_ref_text}" + Style.RESET_ALL)

            Pre_check_response= Local_Model_call(prompt_ref_text,PRE_CHECK_SYSTEM_PROMPT)
            # has_ref, ref_text = process_pre_check_response(Pre_check_response)

            if "None" in Pre_check_response:

                print(Fore.RED +  f"\n --- {prompt_ref_text}\n Result: None Identified" + Style.RESET_ALL)

            else:

                prompt_only_ref_text = f"Reference data:\n {Pre_check_response}"
                print(Fore.YELLOW +  f"\n --- {prompt_only_ref_text}" + Style.RESET_ALL)


                Raw_Res_Ref_data = Local_Model_call(prompt_only_ref_text, SYSTEM_PROMPT_Reference_extraction)
                save_references_to_json(Raw_Res_Ref_data,output_path)


Reference_data = process_references_from_chunks(chunks, doc,output_path)


